In [ ]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import datetime
import numpy as np
import math
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
import scipy.io
from matplotlib.pylab import mpl
from tensorflow import keras

In [ ]:
# split dataset 80% train, 15% validation, 5% dev
def split_dataset(dataset):
    size = dataset.shape[0]
    train_size = size * 2//3
    test_size = size * 1//6

    return dataset[0:train_size, :], dataset[train_size:(train_size + test_size), :], dataset[(train_size + test_size):size, :]

def plot_series(time, series, lab, form='-', start=0, end=None):
    plt.plot(time[start:end], series[start:end], form, label=lab)
    plt.xlabel("t")
    plt.ylabel("Residual")
    #plt.ylim([-25, 25])
    plt.grid(True)

def tf_dataset(series_x, series_y, batch_size, shuffle_buffer, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((series_x, series_y))
    if shuffle:
        ds = ds.cache().shuffle(shuffle_buffer).batch(batch_size).repeat()
    else:
        ds = ds.cache().batch(batch_size).repeat()

    return ds

def create_window_dataset(ds, lb, window_size):
    windowed_dataset = []
    labels = []
    for i in range(window_size, ds.shape[0] + 1):
        windowed_dataset.append(ds[i - window_size:i])
        labels.append(lb[i - 1])

    return np.array(windowed_dataset), np.array(labels)

def get_metrics_result(metrics, true_labels, predicted_labels):
    metrics_result = []
    for metric in metrics:
        metric.reset_states()
        metric.update_state(true_labels, predicted_labels)
        metrics_result.append(metric.result().numpy())

    return metrics_result

In [ ]:
data = scipy.io.loadmat('/kaggle/input/sawtooth/sawtooth.mat')
VA = data['VAs'][1:,:].flatten('F')
IA = data['IAs'][1:,:].flatten('F')
thetam = data['thetaMs'][1:,:].flatten('F')
Dthetam = data['DTheta_ms'][1:,:].flatten('F')
com = data['com_ms'][1:,:].flatten('F')
thetae = 2*math.pi*(4*thetam/2/math.pi-np.floor(4*thetam/2/math.pi))
ki = -0.95765*np.sin(thetae-0*math.pi)

In [ ]:
t=np.arange(0,20,0.001)
t=np.concatenate((t,t,t,t,t,t,t,t,t,t,t,t,t,t,t,t,t,t,t,t,t,t,t,t,t,t,t,t,t,t))
t.shape

In [ ]:
VA=VA.reshape(600000,1)
IA=IA.reshape(600000,1)
thetam=thetam.reshape(600000,1)
Dthetam=Dthetam.reshape(600000,1)
com=com.reshape(600000,1)
ki=ki.reshape(600000,1)
t=t.reshape(600000,1)

In [ ]:
#1test
VA=np.concatenate((VA[200000:,:],VA[:200000,:]))
ki=np.concatenate((ki[200000:,:],ki[:200000,:]))
IA=np.concatenate((IA[200000:,:],IA[:200000,:]))
thetam=np.concatenate((thetam[200000:,:],thetam[:200000,:]))
Dthetam=np.concatenate((Dthetam[200000:,:],Dthetam[:200000,:]))

In [ ]:
#2test
VA=np.concatenate((VA[:200000,:],VA[400000:,:],VA[200000:400000,:]))
IA=np.concatenate((IA[:200000,:],IA[400000:,:],IA[200000:400000,:]))
ki=np.concatenate((ki[:200000,:],ki[400000:,:],ki[200000:400000,:]))
Dthetam=np.concatenate((Dthetam[:200000,:],Dthetam[400000:,:],Dthetam[200000:400000,:]))
thetam=np.concatenate((thetam[:200000,:],thetam[400000:,:],thetam[200000:400000,:]))

In [ ]:
X = np.concatenate((VA,t,ki,Dthetam,IA), axis=1)

# Split dataset

In [ ]:
train_dataset, cross_validation_dataset, dev_dataset = split_dataset(X)

print("Dataset shape: {:s}".format(str(X.shape)))
print("Train dataset shape: {:s}".format(str(train_dataset.shape)))
print("Cross validation dataset shape: {:s}".format(str(cross_validation_dataset.shape)))
print("Dev dataset shape: {:s}".format(str(dev_dataset.shape)))

# Data normalization

In [ ]:
scaler = MinMaxScaler()
scaler = scaler.fit(train_dataset)

train_dataset_normalized = scaler.transform(train_dataset)
cross_validation_dataset_normalized = scaler.transform(cross_validation_dataset)
dev_dataset_normalized = scaler.transform(dev_dataset)

# Create window dataset

In [ ]:
WINDOW_SIZE = 30
BATCH_SIZE = 64

windowed_dataset_train, labels_train = create_window_dataset(train_dataset_normalized[:,0:-1],train_dataset_normalized[:, -1], WINDOW_SIZE)
windowed_dataset_train_0 = windowed_dataset_train[:20000-WINDOW_SIZE+1,:,:]
windowed_dataset_train_2 = np.empty((0,30,4))
for i in range(1,20,1):
    windowed_dataset_train_1 = windowed_dataset_train[i*20000-WINDOW_SIZE+1:(i+1)*20000-WINDOW_SIZE+1,:,:]
    windowed_dataset_train_1 = np.delete(windowed_dataset_train_1,[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28],0)
    windowed_dataset_train_2 = np.concatenate((windowed_dataset_train_2, windowed_dataset_train_1), axis=0)

windowed_dataset_train = np.concatenate((windowed_dataset_train_0, windowed_dataset_train_2), axis=0)

labels_train_0 = labels_train[:20000-WINDOW_SIZE+1]
labels_train_2 = []
for i in range(1,20,1):
    labels_train_1 = labels_train[i*20000-WINDOW_SIZE+1:(i+1)*20000-WINDOW_SIZE+1]
    labels_train_1 = np.delete(labels_train_1,[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28],0)
    labels_train_2 = np.concatenate((labels_train_2, labels_train_1), axis=0)

labels_train = np.concatenate((labels_train_0, labels_train_2), axis=0)

train_set = tf_dataset(windowed_dataset_train, labels_train, BATCH_SIZE, 1000)
unshuffled_train_set = tf_dataset(windowed_dataset_train, labels_train, BATCH_SIZE, 1000, False)

windowed_dataset_validation, labels_validation = create_window_dataset(cross_validation_dataset_normalized[:,0:-1],cross_validation_dataset_normalized[:, -1], WINDOW_SIZE)
windowed_dataset_validation_0 = windowed_dataset_validation[:20000-WINDOW_SIZE+1,:,:]
windowed_dataset_validation_2 = np.empty((0,30,4))
for i in range(1,5,1):
    windowed_dataset_validation_1 = windowed_dataset_validation[i*20000-WINDOW_SIZE+1:(i+1)*20000-WINDOW_SIZE+1,:,:]
    windowed_dataset_validation_1 = np.delete(windowed_dataset_validation_1,[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28],0)
    windowed_dataset_validation_2 = np.concatenate((windowed_dataset_validation_2, windowed_dataset_validation_1), axis=0)

windowed_dataset_validation = np.concatenate((windowed_dataset_validation_0, windowed_dataset_validation_2), axis=0)

labels_validation_0 = labels_validation[:20000-WINDOW_SIZE+1]
labels_validation_2 = []
for i in range(1,5,1):
    labels_validation_1 = labels_validation[i*20000-WINDOW_SIZE+1:(i+1)*20000-WINDOW_SIZE+1]
    labels_validation_1 = np.delete(labels_validation_1,[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28],0)
    labels_validation_2 = np.concatenate((labels_validation_2, labels_validation_1), axis=0)

labels_validation = np.concatenate((labels_validation_0, labels_validation_2), axis=0)
cross_validation_set = tf_dataset(windowed_dataset_validation, labels_validation,BATCH_SIZE, 1000, False)

windowed_dataset_dev, labels_dev = create_window_dataset(dev_dataset_normalized[:,0:-1], dev_dataset_normalized[:, -1], WINDOW_SIZE)
windowed_dataset_dev_0 = windowed_dataset_dev[:20000-WINDOW_SIZE+1,:,:]
windowed_dataset_dev_2 = np.empty((0,30,4))
for i in range(1,5,1):
    windowed_dataset_dev_1 = windowed_dataset_dev[i*20000-WINDOW_SIZE+1:(i+1)*20000-WINDOW_SIZE+1,:,:]
    windowed_dataset_dev_1 = np.delete(windowed_dataset_dev_1,[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28],0)
    windowed_dataset_dev_2 = np.concatenate((windowed_dataset_dev_2, windowed_dataset_dev_1), axis=0)

windowed_dataset_dev = np.concatenate((windowed_dataset_dev_0, windowed_dataset_dev_2), axis=0)

labels_dev_0 = labels_dev[:20000-WINDOW_SIZE+1]
labels_dev_2 = []
for i in range(1,5,1):
    labels_dev_1 = labels_dev[i*20000-WINDOW_SIZE+1:(i+1)*20000-WINDOW_SIZE+1]
    labels_dev_1 = np.delete(labels_dev_1,[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28],0)
    labels_dev_2 = np.concatenate((labels_dev_2, labels_dev_1), axis=0)

labels_dev = np.concatenate((labels_dev_0, labels_dev_2), axis=0)
dev_set = tf_dataset(windowed_dataset_dev, labels_dev, 1, 1000, False)

In [ ]:
windowed_dataset_train.shape

In [ ]:
windowed_dataset_validation.shape

In [ ]:
windowed_dataset_dev.shape

In [ ]:
TRAIN_STEP = math.ceil(windowed_dataset_train.shape[0] / BATCH_SIZE)
VALIDATION_STEP = math.ceil(windowed_dataset_validation.shape[0] / BATCH_SIZE)
DEV_STEP = windowed_dataset_dev.shape[0]

# Define model

In [ ]:
tf.keras.backend.clear_session()

model = tf.keras.models.Sequential([
    tf.keras.layers.Conv1D(filters=64,
                           kernel_size=5,
                           strides=1,
                           padding="causal",
                           activation="relu",
                           input_shape=windowed_dataset_train.shape[-2:]),
    tf.keras.layers.MaxPooling1D(pool_size=2, strides=1, padding="valid"),
    tf.keras.layers.Conv1D(filters=32, kernel_size=3, strides=1, padding="causal", activation="relu"),
    tf.keras.layers.MaxPooling1D(pool_size=2, strides=1, padding="valid"),
    tf.keras.layers.LSTM(128, return_sequences=True),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    #tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(16, activation="relu"),
    #tf.keras.layers.Dropout(0.1),
    tf.keras.layers.Dense(1)
])

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(1e-4,
                                                             decay_steps=100000,
                                                             decay_rate=0.98,
                                                             staircase=False)

model.compile(loss=tf.keras.losses.MeanSquaredError(),
              optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
              metrics=['mae'])
model.summary()

In [ ]:
tf.keras.backend.clear_session()

model = tf.keras.models.Sequential([
    tf.keras.layers.LSTM(128, return_sequences=True, input_shape=windowed_dataset_train.shape[-2:]),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dropout(0.1),
    tf.keras.layers.Dense(1)
])

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(1e-4,
                                                             decay_steps=100000,
                                                             decay_rate=0.98,
                                                             staircase=False)

model.compile(loss=tf.keras.losses.MeanSquaredError(),
              optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
              metrics=['mse'])

model.summary()

# Train model

In [ ]:
class LossCallback(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        training_losses.append(logs['loss'])
        validation_losses.append(logs['val_loss'])
        if epoch % 10 == 0:
            print(f"Epoch {epoch}/{num_epochs}, Loss: {logs['loss']}")

In [ ]:
training_losses = []
validation_losses = []
num_epochs = 200
log_dir = "logs\\" + datetime.datetime.now().strftime("%d-%m-%Y_%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
                                 
history = model.fit(train_set,
                    epochs=num_epochs,
                    steps_per_epoch=TRAIN_STEP,
                    validation_data=cross_validation_set,
                    validation_steps=VALIDATION_STEP,
                    verbose=0,
                    callbacks=[LossCallback()])

In [ ]:
plt.plot(training_losses, label='Training Loss')
plt.plot(validation_losses, label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

## Save model weights

In [ ]:
model.save_weights('weights/inter_model_weights.ckpt')

## Load model weights

In [ ]:
model.load_weights('weights/inter_model_weights.ckpt')

# Predictions

In [ ]:
forecast_train = model.predict(unshuffled_train_set, steps=TRAIN_STEP)
forecast_validation = model.predict(cross_validation_set, steps=VALIDATION_STEP)
forecast_dev = model.predict(dev_set, steps=DEV_STEP)

## Show predictions

In [ ]:
total_forecast = np.concatenate((forecast_train[:,0], forecast_validation[:,0], forecast_dev[:,0]))
total_labels = np.concatenate((labels_train, labels_validation, labels_dev))

plt.figure(figsize=(18, 10))
plot_series(np.arange(total_labels.shape[0]), total_labels, "real value")
plot_series(np.arange(total_labels.shape[0]), total_forecast, "predicted value")
xpositions = [labels_train.shape[0], (labels_train.shape[0] + labels_validation.shape[0])]
for xp in xpositions:
    plt.axvline(x=xp, linestyle='--')
plt.legend(loc='upper left')

metrics = [
    tf.keras.metrics.MeanAbsoluteError(),
    tf.keras.metrics.MeanAbsolutePercentageError(),
    tf.keras.metrics.MeanSquaredError(),
    tf.keras.metrics.RootMeanSquaredError()
]
train_metrics = get_metrics_result(metrics, labels_train, forecast_train[:,0])
train_metrics.insert(0, 'Train')
val_metrics = get_metrics_result(metrics, labels_validation, forecast_validation[:,0])
val_metrics.insert(0, 'Validation')
dev_metrics = get_metrics_result(metrics,labels_dev, forecast_dev[:,0])
dev_metrics.insert(0, 'Dev')

COL_NAMES = ['', 'MAE', 'MAPE', 'MSE', 'RMSE']
metrics_table = pd.DataFrame([train_metrics, val_metrics, dev_metrics], columns=COL_NAMES)
metrics_table = metrics_table.set_index([''])
metrics_table.round(3)

In [ ]:
I_max=max(train_dataset[:,-1])
I_min=min(train_dataset[:,-1])
forecast_train[:,0]=forecast_train[:,0]*(I_max-I_min)+I_min
forecast_validation[:,0]=forecast_validation[:,0]*(I_max-I_min)+I_min
forecast_dev[:,0]=forecast_dev[:,0]*(I_max-I_min)+I_min
labels_train=labels_train*(I_max-I_min)+I_min
labels_validation=labels_validation*(I_max-I_min)+I_min
labels_dev=labels_dev*(I_max-I_min)+I_min

## Predictions: Train data

In [ ]:
plt.figure(figsize=(18, 10))
plot_series(np.arange(labels_train.shape[0]), labels_train, "real value")
plot_series(np.arange(labels_train.shape[0]), forecast_train[:,0], "predicted value")
plt.legend(loc='upper left')

## Predictions: Validation data

In [ ]:
plt.figure(figsize=(18, 10))
plot_series(np.arange(labels_validation.shape[0]), labels_validation, "real value")
plot_series(np.arange(labels_validation.shape[0]), forecast_validation[:,0], "predicted value")
plt.legend(loc='upper left')

## Predictions: Dev data

In [ ]:
plt.figure(figsize=(18, 10))
plot_series(np.arange(labels_dev.shape[0]), labels_dev, "real value")
plot_series(np.arange(labels_dev.shape[0]), forecast_dev[:,0], "predicted value")
plt.legend(loc='upper left')

In [ ]:
plt.figure(figsize=(18, 10))
plot_series(np.arange(0.029,20,0.001), labels_dev[:19971], "real value")
plot_series(np.arange(0.029,20,0.001), forecast_dev[:19971,0], "predicted value")
plt.legend(loc='upper left')

In [ ]:
fft1=np.fft.rfft(IA[500000:520000].reshape(-1))
plt.plot(np.arange(0,10001,1),fft1[:10001])

In [ ]:
fft2=np.concatenate((fft1[:5000],np.zeros((5001))))
plt.plot(np.arange(0,10001,1),fft2)

In [ ]:
ifft1=np.fft.irfft(fft2)
ifft1.shape

In [ ]:
plt.figure(figsize=(12, 7))
plot_series(np.arange(0.029,20,0.001),IA[500029:520000],"real value")
plot_series(np.arange(0.029,20,0.001),ifft1[29:],"filtered value")
plot_series(np.arange(0.029,20,0.001), forecast_dev[:19971,0], "predicted value")
#plot_series(np.arange(0,20,0.001), cal[500000:520000], "simulink value")
plt.legend(loc='upper left')

In [ ]:
plt.figure(figsize=(12, 7))
plot_series(np.arange(14.5,15,0.001),IA[514500:515000],"real value")
plot_series(np.arange(14.5,15,0.001),ifft1[14500:15000],"filtered value")
plot_series(np.arange(14.5,15,0.001), forecast_dev[14471:14971,0], "predicted value")
#plot_series(np.arange(14.5,15,0.001), cal[514500:515000], "simulink value")
plt.legend(loc='upper left')

In [ ]:
plt.figure(figsize=(12, 7))
plot_series(np.arange(0.029,20,0.001),ifft1[29:],"filtered value")
plot_series(np.arange(0.029,20,0.001), forecast_dev[:19971,0], "predicted value")
plt.legend(loc='upper left')

In [ ]:
plt.figure(figsize=(12, 7))
plot_series(np.arange(14.5,15,0.001),ifft1[14500:15000],"Actual value")
plot_series(np.arange(14.5,15,0.001), forecast_dev[14471:14971,0], "predicted value")
plt.legend(loc='upper left')

In [ ]:
plt.figure(figsize=(12, 7))
plot_series(np.arange(5,7.5,0.001),ifft1[5000:7500],"Actual value")
plot_series(np.arange(5,7.5,0.001), forecast_dev[4971:14571,0], "predicted value")
plt.legend(loc='upper left')

In [ ]:
#residual between filtered and predicted
plt.figure(figsize=(12, 7))
plot_series(np.arange(0.029,20,0.001),ifft1[29:]-forecast_dev[:19971,0],"residual")

In [ ]:
#residual between real and predicted
plt.figure(figsize=(12, 7))
#plot_series(np.arange(0.029,20,0.001),forecast_dev[:19971,0]-labels_dev[:19971],"residual between real and predicted")
plot_series(np.arange(0.029,20,0.001),ifft1[29:]-forecast_dev[:19971,0],"residual between actual and predicted value")
plt.legend(loc='upper left')

In [ ]:
#(real,filtered)
from math import sqrt
ifft_2=[]
for i in range(5):
    fft=np.fft.rfft(IA[i*20000+500000:(i+1)*20000+500000].reshape(-1))
    fft1=np.concatenate((fft[:5000],np.zeros((5001))))
    ifft1=np.fft.irfft(fft1)
    ifft_2 = np.concatenate((ifft_2, ifft1[29:]), axis=0)
mse_I = mean_squared_error(labels_dev,ifft_2)
print(mse_I)
print(sqrt(mse_I))

In [ ]:
#(predicted,filtered)
ifft_3=[]
for i in range(5):
    fft=np.fft.rfft(IA[i*20000+500000:(i+1)*20000+500000].reshape(-1))
    fft1=np.concatenate((fft[:5000],np.zeros((5001))))
    ifft1=np.fft.irfft(fft1)
    ifft_3 = np.concatenate((ifft_3, ifft1[29:]), axis=0)
mse_I = mean_squared_error(forecast_dev[:,0],ifft_3)
print(mse_I)
print(sqrt(mse_I))

In [ ]:
#(real,predicted)
mse_I = mean_squared_error(labels_dev,forecast_dev[:,0])
print(mse_I)
print(sqrt(mse_I))

In [ ]:
total_forecast = np.concatenate((forecast_train[:,0], forecast_validation[:,0], forecast_dev[:,0]))
total_labels = np.concatenate((labels_train, labels_validation, labels_dev))

In [ ]:
#residual between predicted value and real value
I_pre_max=[]
total1=0
for i in range(30):
    a=max(np.abs(total_forecast[i*19971:(i+1)*19971]-total_labels[i*19971:(i+1)*19971]))
    I_pre_max.append(a)
    total1=total1+a
print(total1/30)
print(I_pre_max)

In [ ]:
#residual between predicted value and filtered value
b=[]
total2=0
for i in range(30):
    fft=np.fft.rfft(IA[i*20000:(i+1)*20000].reshape(-1))
    fft1=np.concatenate((fft[:5000],np.zeros((5001))))
    ifft1=np.fft.irfft(fft1)
    a=max(np.abs(total_forecast[i*19971:(i+1)*19971]-ifft1[29:]))
    b.append(a)
    total2=total2+a
print(total2/30)
print(b)

In [ ]:
#residual between real value and filtered value
c=[]
total3=0
for i in range(30):
    fft=np.fft.rfft(IA[i*20000:(i+1)*20000].reshape(-1))
    fft1=np.concatenate((fft[:5000],np.zeros((5001))))
    ifft1=np.fft.irfft(fft1)
    a=max(np.abs(total_labels[i*19971:(i+1)*19971]-ifft1[29:]))
    c.append(a)
    total3=total3+a
print(c)
print(total3/30)